In [14]:
import pandas as pd
import numpy as np

# 1. Inspect lines to find where actual column headers start
with open("pasum_pub.radon.txt", "r") as f:
    lines = f.readlines()

header_idx = None
for i, line in enumerate(lines):
    if line.startswith("well_identifier"):
        header_idx = i
        break

# 2. Read the tab-delimited data (skipping metadata comments)
# Note: Line immediately after header is formatting metadata ('10s\t18s...'), skip it
df = pd.read_csv(
    "pasum_pub.radon.txt",
    sep="\t",
    skiprows=header_idx,
    na_values=["", " "],
    low_memory=False
)
df = df.iloc[1:].copy()  # drop the '10s', '18s' format type row

# 3. Clean and standardize column names
rename_map = {
    "well_identifier": "well_id",
    "sampling_date": "sample_date",
    "R_82303_Radon-222": "radon_remark",
    "V_82303_Radon-222": "radon_value",
    "R_76002_Radon-222_2-sigma_combined_uncertainty": "uncert_remark",
    "V_76002_Radon-222_2-sigma_combined_uncertainty": "uncert_2sigma",
    "R_99327_Radon-222_sample-specific_minimum_detectable_concentration": "mdc_remark",
    "V_99327_Radon-222_sample-specific_minimum_detectable_concentration": "mdc_value",
}
df.rename(columns=rename_map, inplace=True)

# 4. Extract aquifer group (e.g., 'GLAC', 'SWPA', 'PBVR')
df["aquifer"] = df["well_id"].str.extract(r"^([A-Z]+)")

# 5. Type conversions
df["radon_value"] = pd.to_numeric(df["radon_value"], errors="coerce")
df["uncert_2sigma"] = pd.to_numeric(df["uncert_2sigma"], errors="coerce")

# Convert 2-sigma combined uncertainty to 1-sigma standard error
df["sigma_obs"] = df["uncert_2sigma"] / 2.0

# 6. Flag detection status:
# '-' = detected, '<' = left-censored at detection limit, NaN = unmeasured
df["status"] = "missing"
df.loc[df["radon_remark"] == "-", "status"] = "detected"
df.loc[df["radon_remark"] == "<", "status"] = "censored"

# Handle physical anomalies: radiation counts with negative noise artifacts set to small positive
df.loc[(df["status"] == "detected") & (df["radon_value"] <= 0), "radon_value"] = 1.0

# 7. Filter down to records ready for modeling
cleaned_df = df[[
    "well_id", "aquifer", "sample_date", "status", 
    "radon_value", "sigma_obs"
]].reset_index(drop=True)

print(cleaned_df["status"].value_counts())
print(cleaned_df.head())

df_radon = cleaned_df

OSError: [Errno 116] Stale file handle: 'pasum_pub.radon.txt'

In [13]:
df_sites = pd.read_csv(
    "pasum_pub.wellsites.txt", 
    sep="\t", 
    skiprows=skip_sites,
    low_memory=False
)
df_sites.head(5)

OSError: [Errno 116] Stale file handle: 'pasum_pub.wellsites.txt'

KeyError: 'latitude'